# ML: demand forecast (LightGBM + MLflow, Unity Catalog registry)
One global model forecasts daily demand per store x item, **28 days ahead** (ADR 0002).
- **Target = censoring-corrected demand**: `units_sold + est_lost_units` from Gold, so stockout days do not teach the model that demand was low
- **direct multi-horizon** model: features from the forecast origin + horizon `h` (1..28) + the target day's calendar, so the whole 28-day horizon is scored at once with the latest data (no recursion, no leakage)
- two controls trained the same way: a model on **raw sales** and a 28-day moving average (what a planner would use)
- graded against the generator's **true demand** (ground truth), not only against our own estimate
- the corrected model is registered in Unity Catalog as `<catalog>.ml.demand_forecaster`, alias `champion`, only if it beats the baseline

In [0]:
%pip install -q lightgbm==4.6.0

In [0]:
%run ./_common

In [0]:
NB = "50_forecast_train"
dbutils.widgets.text("train_days", "182")
TRAIN_DAYS = int(dbutils.widgets.get("train_days"))   # history used for training (kept small: driver memory)
H = 28                                                # forecast horizon (days)
MODEL_NAME = f"{CAT}.ml.demand_forecaster"
CAL_PATH = f"/Volumes/{CAT}/bronze/reference/calendar.csv"
TRUTH_PATH = f"/Volumes/{CAT}/ops/ground_truth/lost_sales.parquet"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.ml")

import numpy as np, pandas as pd, lightgbm as lgb

## 1) Daily feature table `gold.demand_features`
Every feature on day `d` summarises history **up to and including `d`**. A forecast made at origin `o` reads the row
for `o` and predicts `o+1 .. o+28`: the model always uses the latest data, like the moving-average baseline does.
Range windows on a day index (not row windows), so a missing day cannot shift a window.

In [0]:
fact = (spark.table(f"{CAT}.gold.fact_item_day")
          .withColumn("y", F.col("units_sold") + F.col("est_lost_units"))          # corrected demand (target)
          .withColumn("day_idx", F.datediff("snapshot_date", F.lit("2011-01-01").cast("date"))))

cal = (spark.read.option("header", True).csv(CAL_PATH)
         .select(F.to_date("date").alias("snapshot_date"),
                 F.col("wday").cast("int").alias("wday"),
                 F.col("month").cast("int").alias("month"),
                 F.col("event_name_1").isNotNull().cast("int").alias("is_event"),
                 *[F.col(f"snap_{s}").cast("int").alias(f"snap_{s}") for s in ["CA", "TX", "WI"]]))

def codes(col):
    """Stable integer code per id (LightGBM categorical feature)."""
    return fact.select(col).distinct().withColumn(f"{col.replace('_id', '')}_code",
                                                  F.dense_rank().over(Window.orderBy(col)) - 1)

w = Window.partitionBy("store_id", "item_id").orderBy("day_idx")
wd = Window.partitionBy("store_id", "item_id", "wday").orderBy("day_idx")
def hist(c, p):
    """History of column c up to and including the row's day."""
    return {f"{p}_last": F.col(c),
            f"{p}_m7": F.avg(c).over(w.rangeBetween(-6, 0)),
            f"{p}_m28": F.avg(c).over(w.rangeBetween(-27, 0)),
            f"{p}_sd28": F.stddev(c).over(w.rangeBetween(-27, 0)),
            f"{p}_m91": F.avg(c).over(w.rangeBetween(-90, 0)),
            f"{p}_zero28": F.avg((F.col(c) == 0).cast("int")).over(w.rangeBetween(-27, 0)),
            f"{p}_dow4": F.avg(c).over(wd.rangeBetween(-21, 0))}   # same weekday, last 4 weeks
def dow_offsets(p):
    """Same-weekday means of the last 7 days, so any target weekday can be looked up from the origin row."""
    return {f"{p}_dowoff{k}": F.avg(f"{p}_dow4").over(w.rangeBetween(-k, -k)) for k in range(7)}

feat = (fact
        .join(cal, "snapshot_date", "left")
        .join(codes("store_id"), "store_id").join(codes("item_id"), "item_id")
        .join(codes("dept_id"), "dept_id").join(codes("cat_id"), "cat_id")
        .withColumn("snap", F.expr("CASE split(store_id, '_')[0] WHEN 'CA' THEN snap_CA WHEN 'TX' THEN snap_TX ELSE snap_WI END"))
        .withColumns(hist("y", "y")).withColumns(hist("units_sold", "s"))
        .withColumns(dow_offsets("y")).withColumns(dow_offsets("s"))
        .drop("snap_CA", "snap_TX", "snap_WI", "y_dow4", "s_dow4"))

feat.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CAT}.gold.demand_features")
print("feature rows:", table_count(f"{CAT}.gold.demand_features"))

## 2) Training set: forecast origins x horizons
Rows = (series, origin, horizon h = 1..28). Features come from the origin day; calendar comes from the target day.
Weekly origins for training (sampled to fit driver memory), one origin for validation, one for test:
`... train origins | valid origin = cutoff - 28 | test origin = cutoff`.

In [0]:
FT = spark.table(f"{CAT}.gold.demand_features")
min_d, max_d = FT.agg(F.min("snapshot_date"), F.max("snapshot_date")).first()
cutoff = max_d - pd.Timedelta(days=H)                 # test origin: forecast the last 28 known days
valid_origin = cutoff - pd.Timedelta(days=H)          # valid targets end exactly at the test origin
first_origin = min_d + pd.Timedelta(days=91)          # every history window is full from here
train_origins = []
o = valid_origin - pd.Timedelta(days=H)               # train targets end before the valid window starts
while o >= first_origin and len(train_origins) * 7 < TRAIN_DAYS:
    train_origins.append(o); o -= pd.Timedelta(days=7)
assert train_origins, f"not enough history: {min_d} .. {max_d}"
print(f"train origins {train_origins[-1]} .. {train_origins[0]} ({len(train_origins)}) | valid origin {valid_origin} | test origin {cutoff}")

P = ["last", "m7", "m28", "sd28", "m91", "zero28"]
CATS = ["store_code", "item_code", "dept_code", "cat_code"]
BASE = CATS + ["h", "wday", "month", "is_event", "snap", "avg_price"]
Y_FEATS = BASE + [f"y_{p}" for p in P] + ["y_dow"]
S_FEATS = BASE + [f"s_{p}" for p in P] + ["s_dow"]

tgt = (FT.select(F.col("snapshot_date").alias("target_date"), "store_code", "item_code",
                 F.col("y").alias("y_target"), F.col("units_sold").alias("s_target"), "is_stockout",
                 *[F.coalesce(F.col(c), F.lit(0)).cast("int").alias(c) for c in ["wday", "month", "is_event", "snap"]]))
hz = spark.range(1, H + 1).select(F.col("id").cast("int").alias("h"))

def build(origins, frac=1.0):
    org = (FT.where(F.col("snapshot_date").isin(origins))
             .select(F.col("snapshot_date").alias("origin"), *CATS, "avg_price",
                     *[f"{t}_{p}" for t in "ys" for p in P], *[f"{t}_dowoff{k}" for t in "ys" for k in range(7)]))
    x = (org.crossJoin(hz)
            .withColumn("target_date", F.date_add("origin", F.col("h")))
            .join(tgt, ["target_date", "store_code", "item_code"]))
    pick = ((7 - F.col("h") % 7) % 7) + 1             # which of the last 7 days has the target's weekday
    for t in "ys":
        x = x.withColumn(f"{t}_dow", F.element_at(F.array(*[f"{t}_dowoff{k}" for k in range(7)]), pick))
    if frac < 1:
        x = x.sample(fraction=frac, seed=42)
    num = [c for c in set(Y_FEATS + S_FEATS + ["y_target", "s_target"]) if c not in CATS + ["h", "wday", "month", "is_event", "snap"]]
    pdf = x.select("target_date", "is_stockout", *CATS, "h", "wday", "month", "is_event", "snap",
                   *[F.col(c).cast("float") for c in num]).toPandas()
    pdf["target_date"] = pd.to_datetime(pdf["target_date"])
    return pdf

n_series = FT.select("store_code", "item_code").distinct().count()
frac = min(1.0, 4_000_000 / (n_series * len(train_origins) * H))      # cap the training set at ~4M rows
tr = build(train_origins, frac)
va = build([valid_origin])
te = build([cutoff])
ids = FT.select("store_id", "item_id", "cat_id", "store_code", "item_code").distinct().toPandas()
te = te.merge(ids, on=["store_code", "item_code"]).rename(columns={"target_date": "snapshot_date"})
print(f"rows: train {len(tr):,} (sample {frac:.2f}) | valid {len(va):,} | test {len(te):,}")

## 3) True demand for the test window (ground truth)
The generator recorded true lost units; true demand = units sold + true lost units. Baselines = 28-day mean at the origin.

In [0]:
truth = (spark.read.parquet(TRUTH_PATH)
           .withColumn("snapshot_date", F.to_date("date"))
           .where(F.col("snapshot_date") > F.lit(cutoff))
           .groupBy("snapshot_date", "store_id", "item_id").agg(F.sum("lost").alias("true_lost"))
           .toPandas())
truth["snapshot_date"] = pd.to_datetime(truth["snapshot_date"])
te = te.merge(truth, on=["snapshot_date", "store_id", "item_id"], how="left")
te["true_lost"] = te["true_lost"].fillna(0).astype("float32")
te["units_sold"] = te["s_target"]
te["y"] = te["y_target"]
te["y_true"] = te["units_sold"] + te["true_lost"]
te["ma28_sales"] = te["s_m28"].fillna(0)          # what a planner would forecast from sales
te["ma28_corrected"] = te["y_m28"].fillna(0)      # same, on our corrected demand
print(f"test true demand {te.y_true.sum():,.0f} units, of which lost to stockouts {te.true_lost.sum():,.0f}")

## 4) Train, evaluate, log to MLflow, register the champion
WAPE = sum|actual - forecast| / sum(actual): MAPE is undefined here (most item-days have zero demand). Tweedie loss suits intermittent counts.

In [0]:
import mlflow
from mlflow.models import infer_signature
mlflow.set_registry_uri("databricks-uc")
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/supplychainiq_demand_forecast")

def wape(a, f):
    a, f = np.asarray(a, float), np.asarray(f, float)
    return float(np.abs(a - f).sum() / max(np.abs(a).sum(), 1e-9))

def bias(a, f):
    return float(np.asarray(f, float).sum() / max(np.asarray(a, float).sum(), 1e-9) - 1)

PARAMS = dict(objective="tweedie", tweedie_variance_power=1.1, learning_rate=0.08, num_leaves=63,
              min_data_in_leaf=100, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              lambda_l2=1.0, seed=42, verbose=-1)

def fit(target, feats):
    dtr = lgb.Dataset(tr[feats], tr[target], categorical_feature=CATS, free_raw_data=False)
    dva = lgb.Dataset(va[feats], va[target], categorical_feature=CATS, reference=dtr)
    return lgb.train(PARAMS, dtr, num_boost_round=800, valid_sets=[dva],
                     callbacks=[lgb.early_stopping(50, verbose=False)])

def metrics(pred):
    so = te.is_stockout.astype(bool).to_numpy()
    store_day = te.assign(p=pred).groupby(["store_id", "snapshot_date"])[["y_true", "p"]].sum()
    out = {"wape_true_item_day": wape(te.y_true, pred), "bias_true": bias(te.y_true, pred),
           "wape_true_store_day": wape(store_day.y_true, store_day.p),
           "wape_true_stockout_days": wape(te.y_true[so], pred[so])}
    for c in sorted(te.cat_id.unique()):
        m = (te.cat_id == c).to_numpy()
        out[f"wape_true_{c}"] = wape(te.y_true[m], pred[m])
    return out

results = {}
for name, pred in [("ma28_sales", te.ma28_sales.to_numpy()), ("ma28_corrected", te.ma28_corrected.to_numpy())]:
    with mlflow.start_run(run_name=f"baseline_{name}"):
        mlflow.log_params({"model": name, "horizon": H, "cutoff": str(cutoff)})
        results[name] = metrics(pred); mlflow.log_metrics(results[name])

models = {}
for name, target, feats in [("lgbm_sales_target", "s_target", S_FEATS), ("lgbm_corrected_target", "y_target", Y_FEATS)]:
    with mlflow.start_run(run_name=name) as run:
        m = fit(target, feats)
        pred = np.clip(m.predict(te[feats], num_iteration=m.best_iteration), 0, None)
        te[f"pred_{name}"] = pred.astype("float32")
        results[name] = metrics(pred)
        mlflow.log_params({**PARAMS, "model": name, "target": target, "horizon": H, "cutoff": str(cutoff),
                           "train_origins": len(train_origins), "train_rows": len(tr), "best_iteration": m.best_iteration})
        mlflow.log_metrics(results[name])
        imp = pd.Series(m.feature_importance("gain"), index=feats).sort_values(ascending=False)
        mlflow.log_dict(imp.round(1).to_dict(), "feature_importance_gain.json")
        sig = infer_signature(te[feats].head(1000), pred[:1000])
        info = mlflow.lightgbm.log_model(m, "model", signature=sig, input_example=te[feats].head(5))
        models[name] = (m, info.model_uri, feats)

summary = pd.DataFrame(results).T[["wape_true_item_day", "wape_true_store_day", "wape_true_stockout_days", "bias_true"]]
display(summary.round(4).reset_index(names="model"))

In [0]:
from mlflow import MlflowClient
champ = "lgbm_corrected_target"
best_baseline = min(results["ma28_sales"]["wape_true_item_day"], results["ma28_corrected"]["wape_true_item_day"])
beats = results[champ]["wape_true_item_day"] < best_baseline
if beats:
    mv = mlflow.register_model(models[champ][1], MODEL_NAME)
    MlflowClient().set_registered_model_alias(MODEL_NAME, "champion", mv.version)
    print(f"registered {MODEL_NAME} v{mv.version} as @champion")
else:
    print("champion NOT updated: model did not beat the moving-average baseline")

## 5) Outputs for the dashboard
- `gold.forecast_eval`: test-window forecasts vs true demand, per store x item x day
- `gold.forecast_metrics`: one row per model per run (history of model quality)

In [0]:
cols = ["snapshot_date", "store_id", "item_id", "cat_id", "is_stockout", "units_sold", "y", "y_true",
        "ma28_sales", "pred_lgbm_sales_target", "pred_lgbm_corrected_target"]
(spark.createDataFrame(te[cols].rename(columns={"y": "y_corrected"}))
      .withColumn("snapshot_date", F.to_date("snapshot_date"))
      .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CAT}.gold.forecast_eval"))

mrows = pd.DataFrame(results).T.reset_index(names="model")
mrows["cutoff"] = str(cutoff)
(spark.createDataFrame(mrows).withColumn("run_ts", F.current_timestamp())
      .write.mode("append").option("mergeSchema", "true").saveAsTable(f"{CAT}.gold.forecast_metrics"))
print("forecast_eval and forecast_metrics written")

In [0]:
r = results
dq_check("forecast_beats_ma28_baseline", r["lgbm_corrected_target"]["wape_true_item_day"], beats,
         f"< {best_baseline:.4f} (best moving-average baseline)")
dq_check("corrected_target_beats_sales_target", r["lgbm_corrected_target"]["wape_true_item_day"],
         r["lgbm_corrected_target"]["wape_true_item_day"] <= r["lgbm_sales_target"]["wape_true_item_day"],
         f"<= {r['lgbm_sales_target']['wape_true_item_day']:.4f} (raw-sales model)", hard=False)
dq_check("forecast_bias_within_10pct", r["lgbm_corrected_target"]["bias_true"],
         abs(r["lgbm_corrected_target"]["bias_true"]) <= 0.10, "|bias| <= 0.10", hard=False)
dq_finish(NB)